# Pipeline de Treinamento da ResNet-18 Escalável (Google Colab)

Este notebook executa o treinamento do classificador de folhas de tomate usando uma ResNet-18 personalizada desenvolvida do zero (sem pesos pré-treinados), suportando:
- **Fator de Capacidade ($\alpha$)**: Redimensiona a largura das camadas convolucionais (12.5%, 25%, 50% e 100% de capacidade).
- **Inicialização Xavier/Glorot**: Aplicada em todas as camadas lineares e convolucionais.
- **Orçamento Fixo por Steps**: Treinamento delimitado por atualizações de gradiente fixas (ex: 10.000 a 20.000 steps) em vez de épocas tradicionais.
- **Learning Rate Warmup + Cosine Annealing**: warmup linear nos primeiros 5% do treinamento com decaimento cosseno nos 95% restantes.
- **Execução Reprodutível Multisseed**: Executa o treinamento 5 vezes sob seeds distintas e exporta as métricas de teste calculadas em `global_test` para um arquivo CSV.

## Passo 1: Instalação de Dependências e Configuração do Ambiente

In [ ]:
# Instala dependências básicas caso necessário
!pip install -q scikit-learn pillow numpy torch torchvision

import os
import csv
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, precision_score, accuracy_score

print("✓ Bibliotecas importadas com sucesso.")

## Passo 2: Seleção do Hardware de Execução
Compatibilidade integrada com NVIDIA CUDA, AMD DirectML e CPU.

In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        print("[*] Hardware Ativo: CUDA (NVIDIA GPU) detectado no Colab.")
        return torch.device("cuda")
    
    try:
        import torch_directml
        if torch_directml.is_available():
            print("[*] Hardware Ativo: DirectML (AMD/Intel GPU) detectado.")
            return torch_directml.device()
    except ImportError:
        pass
    
    print("[*] Hardware Ativo: CPU detectado.")
    return torch.device("cpu")

device = get_device()

## Passo 3: Definição da Classe de Dataset
Dataset agnóstico e ordenado alfabeticamente para evitar divergências de splits entre diferentes SOs.

In [ ]:
class TomatoDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        self.classes = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG", "*.JPEG")
        for cls_name in self.classes:
            cls_dir = root_dir / cls_name
            class_images = []
            for ext in extensions:
                class_images.extend(list(cls_dir.glob(ext)))
            for img_path in sorted(list(set(class_images))):
                self.image_paths.append(img_path)
                self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

## Passo 4: Implementação da ResNet-18 Escalável
Modelo instanciado do zero suportando o fator de largura $\alpha$ e inicialização Xavier.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential( 
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ScalableResNet18(nn.Module):
    def __init__(self, num_classes: int = 10, alpha: float = 1.0):
        super(ScalableResNet18, self).__init__()
        self.in_planes = int(64 * alpha)
        
        self.conv1 = nn.Conv2d(3, self.in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_planes)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(BasicBlock, int(64 * alpha), 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, int(128 * alpha), 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, int(256 * alpha), 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, int(512 * alpha), 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(int(512 * alpha) * BasicBlock.expansion, num_classes)

        self._init_weights()

    def _make_layer(self, block, planes: int, num_blocks: int, stride: int):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

## Passo 5: Funções Auxiliares de Otimização e Avaliação

In [ ]:
def get_lr(step: int, total_steps: int, lr_max: float, warmup_pct: float = 0.05) -> float:
    warmup_steps = int(total_steps * warmup_pct)
    if step <= warmup_steps:
        return lr_max * (step / max(1, warmup_steps))
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * lr_max * (1.0 + math.cos(math.pi * progress))

def calculate_slope(y: List[float]) -> float:
    n = len(y)
    if n <= 1:
        return 0.0
    x = np.arange(n)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x**2)
    sum_xy = np.sum(x * y)
    
    denominator = (n * sum_xx) - (sum_x**2)
    if denominator == 0:
        return 0.0
    slope = ((n * sum_xy) - (sum_x * sum_y)) / denominator
    return float(slope)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def evaluate(model: nn.Module, dataloader: DataLoader, device: torch.device) -> Tuple[float, float, float]:
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    acc = accuracy_score(all_targets, all_preds)
    test_error = 1.0 - acc
    f1 = f1_score(all_targets, all_preds, average="macro")
    precision = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    
    return test_error, f1, precision

## Passo 6: Loop de Treinamento por Steps

In [ ]:
def train_single_run(
    seed: int,
    train_dir: Path,
    test_dir: Path,
    alpha: float,
    total_steps: int,
    batch_size: int,
    lr_max: float,
    weight_decay: float,
    img_size: int,
    device: torch.device,
    output_csv: str
) -> Tuple[int, float, float, float, float, float]:

    set_seed(seed)

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = TomatoDataset(train_dir, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

    # Check if the training dataset or loader is effectively empty and skip if so
    if len(train_dataset) == 0 or len(train_loader) == 0:
        print(f"  [WARN] Skipping training for Capacity: {alpha:.3f}, Subset: {train_dir.name}, Seed: {seed} - Dataset ou DataLoader vazio. len(train_dataset)={len(train_dataset)}, len(train_loader)={len(train_loader)}.")
        # Initialize a dummy model to calculate total_params, even if not trained
        model_for_params = ScalableResNet18(num_classes=10, alpha=alpha)
        total_params = sum(p.numel() for p in model_for_params.parameters())
        # Return dummy values for empty dataset runs:
        # dataset_size, total_params, loss_slope_final, test_error, f1_score_macro, precision_macro
        return 0, total_params, 0.0, 1.0, 0.0, 0.0


    test_dataset = TomatoDataset(test_dir, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    dataset_size = len(train_dataset)

    model = ScalableResNet18(num_classes=10, alpha=alpha).to(device)
    total_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    train_iter = iter(train_loader)
    loss_history = []

    model.train()
    for step in range(1, total_steps + 1):
        try:
            inputs, targets = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            inputs, targets = next(train_iter)

        inputs, targets = inputs.to(device), targets.to(device)

        current_lr = get_lr(step, total_steps, lr_max)
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    slope_window = int(total_steps * 0.1)
    final_losses = loss_history[-slope_window:]
    loss_slope = calculate_slope(final_losses)

    test_error, f1, precision = evaluate(model, test_loader, device)

    # Append to CSV safely
    csv_file = Path(output_csv)
    fieldnames = ["seed", "tamanho_dataset", "fracao_parametros_modelo", "total_parametros", "loss_slope_final", "test_error", "f1_score_macro", "precision_macro"]
    file_exists = csv_file.exists()
    with open(csv_file, mode="a" if file_exists else "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "seed": seed,
            "tamanho_dataset": dataset_size,
            "fracao_parametros_modelo": alpha,
            "total_parametros": total_params,
            "loss_slope_final": f"{loss_slope:.6e}",
            "test_error": f"{test_error:.6f}",
            "f1_score_macro": f"{f1:.6f}",
            "precision_macro": f"{precision:.6f}"
        })

    return dataset_size, total_params, loss_slope, test_error, f1, precision

## Passo 7: Orquestração das Execuções (Seeds e Capacidades)
Defina os caminhos e parâmetros abaixo antes de rodar o treinamento.

In [ ]:
# Configurações dos experimentos
TRAIN_DIR = Path("/content/dataset_processado/train_fase_1") # Ajuste para a sua pasta de treino no Colab
TEST_DIR = Path("/content/dataset_processado/global_test")   # Ajuste para a sua pasta de teste
ALPHA = 1.0          # Modifique para 0.125 (12.5%), 0.25 (25%), 0.5 (50%) ou 1.0 (100%)
STEPS = 10000        # Orçamento de steps (entre 10.000 e 20.000)
BATCH_SIZE = 64      # Cravado em 64
LR_MAX = 0.002       # LR inicial de 2 x 10^-3
WEIGHT_DECAY = 1e-4  # Decaimento de pesos
IMG_SIZE = 128       # Dimensão da imagem
SEEDS = [42, 43, 44, 45, 46]
OUTPUT_CSV = "colab_training_results.csv"

results = []

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    print("[!] ATENÇÃO: Verifique se os caminhos das pastas de treino e teste foram montados ou criados corretamente.")
else:
    for seed in SEEDS:
        dataset_size, total_params, loss_slope, test_error, f1, precision = train_single_run(
            seed=seed,
            train_dir=TRAIN_DIR,
            test_dir=TEST_DIR,
            alpha=ALPHA,
            total_steps=STEPS,
            batch_size=BATCH_SIZE,
            lr_max=LR_MAX,
            weight_decay=WEIGHT_DECAY,
            img_size=IMG_SIZE,
            device=device
        )
        
        results.append({
            "seed": seed,
            "tamanho_dataset": dataset_size,
            "fracao_parametros_modelo": ALPHA,
            "total_parametros": total_params,
            "loss_slope_final": f"{loss_slope:.6e}",
            "test_error": f"{test_error:.6f}",
            "f1_score_macro": f"{f1:.6f}",
            "precision_macro": f"{precision:.6f}"
        })

    # Salvar resultados em CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[+] Treinamento concluído para as 5 seeds. Resultados salvos em '{OUTPUT_CSV}'.")

## Passo 8: Visualizar Métricas de Resultados obtidos

In [ ]:
if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    display(df)
else:
    print("Nenhum arquivo de resultados encontrado ainda.")